# 03 — Battle Simulation: Distributed Monte Carlo with PySpark

**Goal:** generate millions of simulated battles between every pair of Pokémon, using the real damage formula from `src/battle_engine.py`, distributed across Spark.

**Approach:**
1. Build every unique Pokémon pair (`pokemon_id_a < pokemon_id_b` — order doesn't matter, the engine decides who attacks first based on Speed)
2. Replicate each pair N times (Monte Carlo runs — accounts for randomness: crits, damage rolls, speed ties)
3. Apply the battle simulation as a Spark UDF, distributed across all cores
4. Write the results to `data/silver/simulated_battles_silver/`

**Start with `SAMPLE_MODE = True`** to validate the full pipeline on a small subset before running the real job.

In [11]:
import os
import sys
import time

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, BooleanType

spark = (
    SparkSession.builder
    .appName("PokemonBattleSimulation")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

SILVER_PATH = "../data/silver"

# Ship battle_engine.py to Spark's worker processes so the UDF can import it
battle_engine_path = os.path.abspath(os.path.join("..", "src", "battle_engine.py"))
spark.sparkContext.addPyFile(battle_engine_path)

sys.path.append(os.path.abspath(os.path.join("..", "src")))
from battle_engine import simulate_battle

print("Spark session ready, battle_engine.py shipped to workers.")

Spark session ready, battle_engine.py shipped to workers.


In [12]:
df_pokemon = spark.read.parquet(f"{SILVER_PATH}/pokemon_silver")

NEEDED_COLS = ["pokemon_id", "name", "type_1", "type_2", "hp", "attack", "defense", "sp_atk", "sp_def", "speed"]
df_pokemon = df_pokemon.select(NEEDED_COLS)

print(f"Pokemon available: {df_pokemon.count()}")

Pokemon available: 800


## Sample mode toggle

Run once with `SAMPLE_MODE = True` (fast, a few seconds) to confirm everything works end-to-end. Once confirmed, flip to `False` and rerun the whole notebook for the full-scale simulation.

In [13]:
SAMPLE_MODE = False  # set to False for the full-scale run

if SAMPLE_MODE:
    SAMPLE_POKEMON_COUNT = 50
    N_SIMULATIONS = 5
    df_pokemon_active = df_pokemon.orderBy("pokemon_id").limit(SAMPLE_POKEMON_COUNT)
else:
    N_SIMULATIONS = 20
    df_pokemon_active = df_pokemon

print(f"SAMPLE_MODE = {SAMPLE_MODE} | Pokemon in play: {df_pokemon_active.count()} | Simulations per pair: {N_SIMULATIONS}")

SAMPLE_MODE = False | Pokemon in play: 800 | Simulations per pair: 20


## Build unique pairs

In [14]:
p_a = df_pokemon_active.select([F.col(c).alias(f"{c}_a") for c in NEEDED_COLS])
p_b = df_pokemon_active.select([F.col(c).alias(f"{c}_b") for c in NEEDED_COLS])

pairs = p_a.crossJoin(p_b).filter(F.col("pokemon_id_a") < F.col("pokemon_id_b"))

pair_count = pairs.count()
projected_rows = pair_count * N_SIMULATIONS
print(f"Unique pairs: {pair_count:,}")
print(f"Projected total battle rows (pairs x simulations): {projected_rows:,}")

Unique pairs: 319,600
Projected total battle rows (pairs x simulations): 6,392,000


## Monte Carlo replication

`F.sequence` + `F.explode` turns each pair into `N_SIMULATIONS` rows. Each row gets a deterministic seed (derived from the pair's IDs and run number) so results are reproducible — rerunning this notebook gives the exact same simulated battles, not a different random sample each time.

In [15]:
pairs_expanded = (
    pairs
    .withColumn("run_id", F.explode(F.sequence(F.lit(1), F.lit(N_SIMULATIONS))))
    .withColumn(
        "battle_seed",
        (F.col("pokemon_id_a") * F.lit(100000) + F.col("pokemon_id_b") * F.lit(100) + F.col("run_id")).cast("long"),
    )
)

NUM_PARTITIONS = spark.sparkContext.defaultParallelism * 4
pairs_expanded = pairs_expanded.repartition(NUM_PARTITIONS)

print(f"Repartitioned into {NUM_PARTITIONS} partitions for parallel execution.")

Repartitioned into 32 partitions for parallel execution.


## Battle simulation UDF

Wraps `simulate_battle()` from `battle_engine.py`. Takes each Pokémon's stats as individual scalar columns (simpler and more explicit than passing nested structs) and returns a struct with the battle outcome.

In [16]:
battle_result_schema = StructType([
    StructField("winner_id", IntegerType(), False),
    StructField("turns", IntegerType(), False),
    StructField("faster_pokemon_won", BooleanType(), False),
])


def _simulate_battle_udf(
    pid_a, type1_a, type2_a, hp_a, atk_a, def_a, spatk_a, spdef_a, speed_a,
    pid_b, type1_b, type2_b, hp_b, atk_b, def_b, spatk_b, spdef_b, speed_b,
    seed,
):
    pokemon_a = {
        "pokemon_id": pid_a, "type_1": type1_a, "type_2": type2_a, "hp": hp_a,
        "attack": atk_a, "defense": def_a, "sp_atk": spatk_a, "sp_def": spdef_a, "speed": speed_a,
    }
    pokemon_b = {
        "pokemon_id": pid_b, "type_1": type1_b, "type_2": type2_b, "hp": hp_b,
        "attack": atk_b, "defense": def_b, "sp_atk": spatk_b, "sp_def": spdef_b, "speed": speed_b,
    }
    result = simulate_battle(pokemon_a, pokemon_b, seed)
    return (result["winner_id"], result["turns"], result["faster_pokemon_won"])


simulate_battle_udf = F.udf(_simulate_battle_udf, battle_result_schema)
print("UDF registered.")

UDF registered.


In [17]:
battles = (
    pairs_expanded
    .withColumn(
        "battle_result",
        simulate_battle_udf(
            F.col("pokemon_id_a"), F.col("type_1_a"), F.col("type_2_a"), F.col("hp_a"),
            F.col("attack_a"), F.col("defense_a"), F.col("sp_atk_a"), F.col("sp_def_a"), F.col("speed_a"),
            F.col("pokemon_id_b"), F.col("type_1_b"), F.col("type_2_b"), F.col("hp_b"),
            F.col("attack_b"), F.col("defense_b"), F.col("sp_atk_b"), F.col("sp_def_b"), F.col("speed_b"),
            F.col("battle_seed"),
        ),
    )
    .select(
        "pokemon_id_a",
        "pokemon_id_b",
        "run_id",
        "battle_seed",
        F.col("battle_result.winner_id").alias("winner_id"),
        F.col("battle_result.turns").alias("turns"),
        F.col("battle_result.faster_pokemon_won").alias("faster_pokemon_won"),
    )
    .withColumn("_simulated_at", F.current_timestamp())
)

battles.printSchema()

root
 |-- pokemon_id_a: integer (nullable = true)
 |-- pokemon_id_b: integer (nullable = true)
 |-- run_id: integer (nullable = false)
 |-- battle_seed: long (nullable = true)
 |-- winner_id: integer (nullable = true)
 |-- turns: integer (nullable = true)
 |-- faster_pokemon_won: boolean (nullable = true)
 |-- _simulated_at: timestamp (nullable = false)



## Run the simulation and write to Silver

This is the action that actually triggers the distributed computation (everything above is lazy). Timed so we have a real number to cite ("simulated N million battles in M minutes") — a good portfolio talking point.

In [18]:
OUTPUT_PATH = f"{SILVER_PATH}/simulated_battles_silver"

start = time.time()
battles.write.mode("overwrite").parquet(OUTPUT_PATH)
elapsed = time.time() - start

row_count = spark.read.parquet(OUTPUT_PATH).count()

print(f"Wrote {row_count:,} simulated battles in {elapsed:.1f} seconds ({row_count/elapsed:,.0f} rows/sec).")

Wrote 6,392,000 simulated battles in 133.5 seconds (47,889 rows/sec).


## Sanity check

Quick check that the output makes sense: a Pokémon with a large stat advantage should win the clear majority of its simulated battles.

In [19]:
check = spark.read.parquet(OUTPUT_PATH)
check.show(10)

print("Turn count distribution:")
check.groupBy("turns").count().orderBy("turns").show(10)

print("Faster Pokemon win rate (sanity check — should be well above 50%):")
check.groupBy("faster_pokemon_won").count().show()

+------------+------------+------+-----------+---------+-----+------------------+--------------------+
|pokemon_id_a|pokemon_id_b|run_id|battle_seed|winner_id|turns|faster_pokemon_won|       _simulated_at|
+------------+------------+------+-----------+---------+-----+------------------+--------------------+
|         327|         400|    18|   32740018|      400|    2|             false|2026-08-12 22:24:...|
|         531|         549|    18|   53154918|      549|    2|             false|2026-08-12 22:24:...|
|         164|         387|    14|   16438714|      164|    1|              true|2026-08-12 22:24:...|
|         127|         622|    10|   12762210|      127|    2|              true|2026-08-12 22:24:...|
|         447|         468|     8|   44746808|      447|    1|             false|2026-08-12 22:24:...|
|         233|         648|    14|   23364814|      233|    1|             false|2026-08-12 22:24:...|
|          10|         600|    11|    1060011|      600|    1|           

In [20]:
spark.stop()